# Benchmark GHZ State Preparation

Preparing a Greenberger-Horne-Zeilinger (GHZ) state is a standard benchmark for multi-qubit quantum devices because it directly tests their ability to create and maintain genuinely multipartite entanglement [[1]](#Leibfried2005), [[2]](#Kam2024). From the initial state $| 0 \rangle^{\otimes n}$, the model prepares the superposition
$$
| 0 \rangle^{\otimes n} \rightarrow \frac{| 0 \rangle^{\otimes n} + | 1 \rangle^{\otimes n}}{\sqrt{2}},
$$
whose signature combines concentrated populations in the computational basis with strong global phase coherence [[3]](#Monz2011). Since both features are highly sensitive to noise and control errors, GHZ states provide a simple but powerful way to evaluate the performance of a device beyond single-qubit or pairwise behavior.

***

### The model

We use Classiq's `prepare_ghz_state` function from the open-library, for preparing the GHZ state. Then, we apply a single qubit rotation on each qubit, using an $R$ gate rotation. Executing this model for different rotations allows us to define a scoring function relevant to the GHZ state, as described below.


### The Scoring function


To score GHZ-state preparation, we follow the population-and-coherence fidelity estimator of Monz et al. [[2]](#Monz2011). Ideally, an $n$-qubit GHZ state has the form

$$
\frac{| 0\ldots 0 \rangle_n + | 1\ldots 1 \rangle_n}{\sqrt{2}}.
$$

Its quality is quantified using two contributions. First, we estimate the GHZ populations in the computational basis,

$$
P = p_{0} + p_{2^n-1},
$$

where $p_{0}$ and $p_{2^n-1}$ are the observed probabilities of the all-zero and all-one outcomes.

Second, we estimate the coherence by applying a collective analysis rotation to every qubit. After the GHZ state is prepared, each qubit is rotated along the *same* direction and phase (a $\pi/2$ rotation around an equatorial Bloch-sphere axis set by $\phi$):

$$
\exp\!\left(i\frac{\pi}{4}\sigma_\phi\right),
\qquad
\sigma_\phi = \sigma_x \cos\phi + \sigma_y \sin\phi.
$$


We then vary the phase $\phi$ and measure the parity

$$
\Pi(\phi) = P_{\mathrm{even}}(\phi) - P_{\mathrm{odd}}(\phi),
$$

where $P_{\mathrm{even}}$ and $P_{\mathrm{odd}}$ are the total probabilities of outcomes with an even or odd number of excitations, respectively. The coherence $C$ is defined as the amplitude of the resulting parity oscillation [[3]](#Monz2011).

The final GHZ score is the fidelity estimator

$$
F = \frac{P + C}{2}.
$$

Following [[3]](#Monz2011), values above $F > 0.5$ certify genuine multipartite entanglement.

***
***

In [19]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *

In [20]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project
# reset_benchmark_project()
# ============================================================

In [21]:
from examples.ghz import GHZExample

***
***
## Benchmarking a 3-qubits GHZ

Define a specific example on 3 qubits:

In [22]:
example = GHZExample(problem_size=3)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3DG0e7sEVjjpfF0Eu5VSDFkhLgz


### Set a `ResultCollector` with a file name fixed for the current example

In [23]:
FILENAME = example.default_results_filename

In [24]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [25]:
# Uncomment to clear the data of a previous run
#
# await collector.reset_file()

### Choose on which backend to run 

We can print the list of backends:

In [26]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [27]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [28]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [29]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [30]:
print(
    "=" * 10
    + f"  {example.name}-{example.problem_size} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  ghz-3 (2026-05-04 14:13:12.212698)   ==========


/Users/tomergoldfriend/classiq-library/benchmarking/benchmarks/examples/ghz.py:72: ClassiqDeprecationWarning: 'submit_batch_sample' is deprecated; pass a list of parameter dicts to 'submit_sample' instead.
  job = es.submit_batch_sample(


2026-05-04 14:13:15.564551: Submit ghz-3 for Classiq - simulator
2026-05-04 14:13:16.296405: Completed ghz-3 for Classiq - simulator --> Score {'score': 1.0, 'execution_time': 0.0146568}
Rc files read:
  NONE
Latexmk: This is Latexmk, John Collins, 27 Dec. 2024. Version 4.86a.
Force everything to be remade.
Latexmk: applying rule 'pdflatex'...
Rule 'pdflatex':  Reasons for rerun
Category 'other':
  Rerun of 'pdflatex' forced or previously required:
    Reason or flag: 'go_mode'

------------
Run number 1 of rule 'pdflatex'
------------
------------
Running 'pdflatex  -interaction=nonstopmode -halt-on-error -recorder -output-directory="build"  "report.tex"'
------------
This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./report.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06

/Users/tomergoldfriend/classiq-library/benchmarking/benchmarks/examples/ghz.py:72: ClassiqDeprecationWarning: 'submit_batch_sample' is deprecated; pass a list of parameter dicts to 'submit_sample' instead.
  job = es.submit_batch_sample(


2026-05-04 14:13:19.814701: Submit ghz-3 for Classiq - simulator_statevector
2026-05-04 14:13:20.474638: Completed ghz-3 for Classiq - simulator_statevector --> Score {'score': 1.0, 'execution_time': 0.013249733333333335}
** Report updated: ghz-3 for Classiq - simulator_statevector


In [31]:
await collector.print_status()

========== (2026-05-04 14:13:21.405880)   ==========
ghz-3 | Classiq - simulator | ERROR 
ghz-3 | Classiq - simulator_statevector | COMPLETED | score=1.0000 | time=0.01 min


## References

<a id='Leibfried2005'>[1]</a>: [D. Leibfried, E. Knill, S. Seidelin, J. Britton, R. B. Blakestad, J. Chiaverini, D. Hume, W. M. Itano, J. D. Jost, C. Langer, R. Ozeri, R. Reichle, D. J. Wineland. "Creation of a six-atom Schrödinger cat state". Nature 438 (2005).](https://www.nist.gov/publications/creation-six-atom-schrodinger-cat-state)

<a id='Kam2024'>[2]</a>: [J. F. Kam, H. Kang, C. D. Hill, G. J. Mooney, L. C. L. Hollenberg. "Characterization of entanglement on superconducting quantum computers of up to 414 qubits". Physical Review Research 6, 033155 (2024).](https://arxiv.org/abs/2312.15170)

<a id='Monz2011'>[3]</a>: [T. Monz, P. Schindler, J. T. Barreiro, M. Chwalla, D. Nigg, W. A. Coish, M. Harlander, W. H\"ansel, M. Hennrich, R. Blatt. "14-qubit entanglement: creation and coherence". arXiv:1009.6126 [quant-ph] (2010).](https://arxiv.org/abs/1009.6126)